# TF-IDF Retrieval

This notebook implements the retrieval component of the retrieval-augmented generation (RAG) pipeline.

## Purpose

TF-IDF is used as a sparse retrieval baseline to retrieve the course-material chunks that are most relevant to a student question. The retrieved chunks can later be passed to an LLM or a sequence-to-sequence model to generate an answer.

## Pipeline

1. Load the preprocessed course-material chunks created in Notebook 03.
2. Build a TF-IDF index from the course-material chunks.
3. Retrieve the top-$k$ chunks for each question using cosine similarity.
4. Evaluate retrieval performance against the annotated source pages.
5. Use the retrieved chunks as context for the LLM and sequence-to-sequence experiments.

- **Train:** retrieved contexts are used as input to the LLM and sequence-to-sequence experiments.
- **Validation:** retrieved contexts are used to tune and evaluate the retrieval and generation pipeline.
- **Test:** retrieved contexts are used for the final, held-out evaluation.

## Data flow

- **Input documents:** `data/processed/chunks_preprocessed.jsonl`
- **Questions:** `data/splits/train.jsonl`, `data/splits/validation.jsonl`, and `data/splits/test.jsonl`
- **Retriever:** TF-IDF with cosine similarity
- **Output:** ranked chunks, retrieved page references, and retrieval metrics

The TF-IDF retriever is fitted on the course-material chunks. The same fitted retriever is then used to retrieve relevant chunks for the train, validation, and test questions.

## RAG Retrieval Pipeline

```text
Notebook 03
Preprocessing and splitting
        |
        v
Preprocessed course chunks
        |
        v
Notebook 04
TF-IDF retriever
        |
        +------------------+
        |                  |
        v                  v
Train questions      Validation questions
retrieved contexts   tune and evaluate
        |
        v
Answer generator
        |
        +------------------+
        |                  |
        v                  v
      LLM              Seq2Seq
        |                  |
        +--------+---------+
                 v
              Answers

Test questions
        |
        v
Final retrieval evaluation
        |
        v
Final LLM and Seq2Seq evaluation
```

In [3]:
import json
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
TOP_K_VALUES = (1, 3, 5, 10)

In [5]:
PROJECT_ROOT = Path.cwd()

PREPROCESSING_NOTEBOOK_PATH = (
    PROJECT_ROOT
    / "03_proccessingSplitting.ipynb"
)

CHUNKS_PREPROCESSED_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "chunks_preprocessed.jsonl"
)

TRAIN_PATH = (
    PROJECT_ROOT
    / "data"
    / "splits"
    / "train.jsonl"
)

VALIDATION_PATH = (
    PROJECT_ROOT
    / "data"
    / "splits"
    / "validation.jsonl"
)

TEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "splits"
    / "test.jsonl"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "data"
    / "retrieval"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

assert PREPROCESSING_NOTEBOOK_PATH.exists()
assert CHUNKS_PREPROCESSED_PATH.exists()
assert TRAIN_PATH.exists()
assert VALIDATION_PATH.exists()
assert TEST_PATH.exists()

print("Notebook 03:", PREPROCESSING_NOTEBOOK_PATH)
print("Chunkovi:", CHUNKS_PREPROCESSED_PATH)
print("Trening skup:", TRAIN_PATH)
print("Validacioni skup:", VALIDATION_PATH)
print("Test skup:", TEST_PATH)

Notebook 03: /home/julijana/Desktop/Student-Question-Answering-from-Course-Materials/03_proccessingSplitting.ipynb
Chunkovi: /home/julijana/Desktop/Student-Question-Answering-from-Course-Materials/data/processed/chunks_preprocessed.jsonl
Trening skup: /home/julijana/Desktop/Student-Question-Answering-from-Course-Materials/data/splits/train.jsonl
Validacioni skup: /home/julijana/Desktop/Student-Question-Answering-from-Course-Materials/data/splits/validation.jsonl
Test skup: /home/julijana/Desktop/Student-Question-Answering-from-Course-Materials/data/splits/test.jsonl


In [6]:
with PREPROCESSING_NOTEBOOK_PATH.open("r", encoding="utf-8") as file:
    notebook_03 = json.load(file)

load_jsonl_source = next(
    cell["source"]
    for cell in notebook_03["cells"]
    if cell.get("cell_type") == "code"
    and any(
        line.startswith("def load_jsonl")
        for line in cell.get("source", [])
    )
)

exec("".join(load_jsonl_source), globals())
print("Funkcija load_jsonl je učitana iz notebooka 03.")

Funkcija load_jsonl je učitana iz notebooka 03.


In [7]:
chunks = load_jsonl(CHUNKS_PREPROCESSED_PATH)

train_data = load_jsonl(TRAIN_PATH)
validation_data = load_jsonl(VALIDATION_PATH)
test_data = load_jsonl(TEST_PATH)

print(f"Broj chunkova: {len(chunks)}")
print(f"Broj pitanja u trening skupu: {len(train_data)}")
print(f"Broj pitanja u validacionom skupu: {len(validation_data)}")
print(f"Broj pitanja u test skupu: {len(test_data)}")

Broj chunkova: 344
Broj pitanja u trening skupu: 100
Broj pitanja u validacionom skupu: 21
Broj pitanja u test skupu: 22


In [19]:
assert chunks

assert all(
    isinstance(chunk.get("processed_text"), str)
    and chunk["processed_text"].strip()
    for chunk in chunks
)

for split_name, data in {
    "train": train_data,
    "validation": validation_data,
    "test": test_data
}.items():

    assert data, f"{split_name} split is empty"

    assert all(
        isinstance(example.get("processed_question"), str)
        and example["processed_question"].strip()
        for example in data
    ), f"Missing processed_question in {split_name}"

In [20]:
chunks_df = pd.DataFrame(chunks)

train_df = pd.DataFrame(train_data)
validation_df = pd.DataFrame(validation_data)
test_df = pd.DataFrame(test_data)

chunks_df[
    ["chunk_id", "pdf_page_start", "pdf_page_end", "processed_text"]
].head(3)

,chunk_id,pdf_page_start,pdf_page_end,processed_text
0,chunk_0001,17,17,Pregled\n\n1.1 Upravljanje kvalitetom softvera...
1,chunk_0002,17,17,"avstvene zaštite. Takođe, softver ima ključnu ..."
2,chunk_0003,17,18,upravljanjem procesima njegove izrade. U proce...


In [ ]:
chunk_ids = chunks_df["chunk_id"].tolist()

chunk_texts = chunks_df["processed_text"].tolist()

print(f"Broj chunkova: {len(chunk_texts)}")
print(f"Broj ID-ova chunkova: {len(chunk_ids)}")

Number of chunks: 344
Number of chunk IDs: 344


In [13]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True,
)

In [14]:
tfidf_matrix = vectorizer.fit_transform(chunk_texts)

print("Oblik matrice TF-IDF :", tfidf_matrix.shape)
print("Velicina rečnika:", len(vectorizer.vocabulary_))
print(f"Broj nenultih vrednosti: {tfidf_matrix.nnz}")

Oblik matrice TF-IDF : (344, 33302)
Velicina rečnika: 33302
Broj nenultih vrednosti: 74041


### Retrieve the most similar chunks

Cosine similarity ranks chunks by their TF-IDF similarity to a question. A chunk is relevant when its PDF page interval overlaps one of the annotated source pages.

In [ ]:
def retrieve_chunks(
    processed_question: str,
    top_k: int = 5
) -> pd.DataFrame:

    question_vector = vectorizer.transform(
        [processed_question]
    )

    if question_vector.nnz == 0:
        return pd.DataFrame(columns=[
            "rank",
            "chunk_id",
            "score",
            "pdf_page_start",
            "pdf_page_end",
            "processed_text",
        ])
    
    similarities = cosine_similarity(
        question_vector,
        tfidf_matrix
    ).flatten()

    top_indices = np.argsort(
        similarities
    )[::-1][:top_k]

    results = []

    for rank, index in enumerate(top_indices, start=1):
        chunk = chunks[index]

        results.append({
            "rank": rank,
            "chunk_id": chunk["chunk_id"],
            "score": float(similarities[index]),
            "pdf_page_start": chunk["pdf_page_start"],
            "pdf_page_end": chunk["pdf_page_end"],
            "processed_text": chunk["processed_text"]
        })

    return pd.DataFrame(results)

In [24]:
example = test_data[1]

print("Original question:")
print(example["question"])

print("\nProcessed question:")
print(example["processed_question"])

print("\nGold source pages:")
print(example["source_pages"])

Original question:
Navesti primere fatalnih posledica pri neispravnom softveru.

Processed question:
Navesti primere fatalnih posledica pri neispravnom softveru.

Gold source pages:
[42, 43, 44, 45, 46]


In [25]:
def overlaps_gold_pages(
    row,
    source_pages
):
    return any(
        row["pdf_page_start"] <= page <= row["pdf_page_end"]
        for page in source_pages
    )

In [26]:
retrieved = retrieve_chunks(
    example["processed_question"],
    top_k=10
)

retrieved["relevant"] = retrieved.apply(
    lambda row: overlaps_gold_pages(
        row,
        example["source_pages"]
    ),
    axis=1
)

retrieved[
    [
        "rank",
        "score",
        "pdf_page_start",
        "pdf_page_end",
        "relevant",
        "processed_text"
    ]
]

,rank,score,pdf_page_start,pdf_page_end,relevant,processed_text
0,1,0.087306,44,44,True,[Greške u softveru]\n\nima se balističke raket...
1,2,0.079853,50,51,False,Literatura\n\n[1] Victor R. Basili i Barry T. ...
2,3,0.042755,131,132,False,[Tehnike testiranja]\n\nmutacija elemenata ne ...
3,4,0.039383,36,37,False,"McGraw-Hill, 2014. isbn: 9780078022128.\n\n[5]..."
4,5,0.035925,49,49,False,[Troškovi usled grešaka u softveru]\n\navljeni...
5,6,0.035018,37,37,False,ke u softveru mogu na različite načine uticati...
6,7,0.033704,64,65,False,"[Testiranje]\n\nanjem, izvođenjem i dokumentov..."
7,8,0.033445,65,65,False,[Testiranje i razvoj softvera]\n\nosti. Ovi pr...
8,9,0.031957,58,61,False,7-3540-6.\n\nIspitna pitanja\n\n8. Odnos verif...
9,10,0.030146,107,108,False,[Tehnike testiranja]\n\njne reči umesto broja....
